# 01 — Crawl Real Corpus (100 models)

**Purpose.** Download real benign HuggingFace `pytorch_model.bin` checkpoints for 5 task clusters (20 each), dedupe by SHA-256, and write `seed_manifest.json`. Resumable; re-running skips existing hashes and backfills already-present files.

**Inputs / outputs**
- `scripts/crawl_benign.py`
- `data/crawled/` (cluster dirs + manifest)

**Outputs**
- `data/crawled/seed_manifest.json` (total_models = 100)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
run(["python3", "scripts/crawl_benign.py",
     "--clusters", "text-generation,text-classification,feature-extraction,token-classification,question-answering",
     "--limit-per-cluster", "25", "--max-size", "134217728",
     "--scan-cap", "20000", "--workers", "8", "--out-dir", "data/crawled", "--format", "both"])

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
import json
m = json.load(open("data/crawled/seed_manifest.json"))
print("total_models:", m["summary"]["total_models"])
for c, n in m["summary"]["clusters"].items(): print(f"  {c}: {n}")
assert m["summary"]["total_models"] >= 250, "crawl did not reach 250"

In [ ]:
import os, glob
os.makedirs('real_benign_corpus/all_pt', exist_ok=True)
os.makedirs('real_benign_corpus/all_gguf', exist_ok=True)
linked = 0
for f in sorted(glob.glob('data/crawled/*/*/pytorch_model.bin')):
    repo = os.path.basename(os.path.dirname(f))
    cluster = os.path.basename(os.path.dirname(os.path.dirname(f)))
    dst = os.path.join('real_benign_corpus', 'all_pt', f'{cluster}__{repo}.bin')
    if os.path.exists(dst):
        os.remove(dst)
    os.link(f, dst)
    linked += 1
print(f'linked {linked} PT checkpoints into real_benign_corpus/all_pt/')
linked = 0
for f in sorted(glob.glob('data/crawled/*/*/*.gguf')):
    repo = os.path.basename(os.path.dirname(f))
    cluster = os.path.basename(os.path.dirname(os.path.dirname(f)))
    dst = os.path.join('real_benign_corpus', 'all_gguf', f'{cluster}__{repo}.gguf')
    if os.path.exists(dst):
        os.remove(dst)
    os.link(f, dst)
    linked += 1
print(f'linked {linked} GGUF checkpoints into real_benign_corpus/all_gguf/')

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks'))
from common import run, run_silent, sqlite, show, summary_line
n_pt = run_silent(["bash", "-c", "ls real_benign_corpus/all_pt 2>/dev/null | wc -l"]).stdout.strip()
n_gguf = run_silent(["bash", "-c", "ls real_benign_corpus/all_gguf 2>/dev/null | wc -l"]).stdout.strip()
print("real_benign_corpus/all_pt files:", n_pt)
print("real_benign_corpus/all_gguf files:", n_gguf)
assert int(n_pt) >= 125
assert int(n_gguf) >= 125